# Pulling data from the CELLxGENE Census API, step by step

A learning notebook (not app code). Run one cell at a time with the **scrna** kernel and read the note above each cell. It needs the internet.

**The idea:** Census keeps ~217 million cells in cloud storage (Amazon S3). The `cellxgene_census` package is the *client library*: it sends requests for just the slice you ask for, so you never download everything.

To keep each step quick, this notebook uses **human spleen** (~0.6 million healthy cells) instead of blood (~14 million), and fetches only 50 cells per cell type.

## Step 1: The client library

Installed with pinned versions (cellxgene-census 1.18.0, tiledbsoma 2.3.0) so the code behaves the same next month. Check what is installed:

In [1]:
import cellxgene_census
import tiledbsoma
import numpy as np
import pandas as pd

print('cellxgene_census', cellxgene_census.__version__)
print('tiledbsoma', tiledbsoma.__version__)

cellxgene_census 1.18.0
tiledbsoma 2.3.0


## Step 2: Choose and pin a data release

Census publishes new data regularly. Without a fixed release, the same code can give different answers later. List the releases (`lts` = long-term support, kept for years):

In [2]:
releases = cellxgene_census.get_census_version_directory()
pd.DataFrame([{'name': name, 'build': info.get('release_build'), 'lts': (info.get('flags') or {}).get('lts', False)}
              for name, info in releases.items()]).head(10)

,name,build,lts
0,stable,2025-11-08,True
1,latest,2025-11-17,False
2,2025-11-17,2025-11-17,False
3,2025-11-10,2025-11-10,False
4,2025-11-08,2025-11-08,True
5,2025-11-04,2025-11-04,False
6,2025-01-30,2025-01-30,True
7,2024-07-01,2024-07-01,True
8,2024-05-20,2024-05-20,False
9,2023-12-15,2023-12-15,True


We pin one release by its date. `with ... as census:` opens the connection and closes it automatically at the end. The summary table is tiny, so this is fast.

In [3]:
CENSUS_VERSION = '2025-11-08'   # the current 'stable' release, flagged long-term support

with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    summary = census['census_info']['summary'].read().concat().to_pandas()
summary[['label', 'value']]

,label,value
0,census_schema_version,2.4.0
1,census_build_date,2025-11-08
2,dataset_schema_version,7.0.0
3,total_cell_count,217768036
4,unique_cell_count,125463259


## Step 3: Look before you fetch (cell descriptions only)

Every cell has a row of descriptions (`obs`): tissue, cell type, disease, study. Reading a few of those columns is cheap; gene expression is the expensive part.

- `value_filter` is a question the **server** answers, so only matching rows travel to you.
- `column_names` asks only for the columns you need.
- `is_primary_data == True` drops duplicate copies of cells that appear in more than one dataset.

In [4]:
import time

TISSUE = 'spleen'
query = f"is_primary_data == True and disease == 'normal' and tissue_general == '{TISSUE}'"

start = time.time()
with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    cells = census['census_data']['homo_sapiens'].obs.read(
        value_filter=query,
        column_names=['soma_joinid', 'cell_type', 'dataset_id'],
    ).concat().to_pandas()
print(f'{len(cells):,} cells in {time.time() - start:.0f} s')
cells.head()

577,677 cells in 5 s


,soma_joinid,cell_type,dataset_id,is_primary_data,disease,tissue_general
0,9757343,"CD8-positive, alpha-beta T cell",644a578d-ffdc-446b-9679-e7ab4c919c13,True,normal,spleen
1,9757344,natural killer cell,644a578d-ffdc-446b-9679-e7ab4c919c13,True,normal,spleen
2,9757345,"activated CD8-positive, alpha-beta T cell",644a578d-ffdc-446b-9679-e7ab4c919c13,True,normal,spleen
3,9757346,B cell,644a578d-ffdc-446b-9679-e7ab4c919c13,True,normal,spleen
4,9757347,natural killer cell,644a578d-ffdc-446b-9679-e7ab4c919c13,True,normal,spleen


## Step 4: Decide which cells you want

We keep cell types seen in **at least 100 cells from at least 2 studies** (so one study's labelling can't decide a type), then pick a random sample of each with a fixed seed (same cells every time). `soma_joinid` is each cell's ID inside Census.

In [5]:
per_type = cells.groupby('cell_type').agg(cells=('soma_joinid', 'size'), studies=('dataset_id', 'nunique'))
kept = per_type[(per_type.cells >= 100) & (per_type.studies >= 2)].sort_values('cells', ascending=False)
print(len(per_type), 'cell types;', len(kept), 'pass the rule')
kept

898 cell types; 45 pass the rule


/var/folders/07/3l4k_4wn505bhm3d0hvndqqc0000gn/T/ipykernel_2462/1027850751.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_type = cells.groupby('cell_type').agg(cells=('soma_joinid', 'size'), studies=('dataset_id', 'nunique'))


,cells,studies
cell_type,,
memory B cell,51953,2
B cell,36535,4
fibroblast,34459,3
naive B cell,32660,2
natural killer cell,31412,3
macrophage,24710,6
classical monocyte,22290,3
plasma cell,21225,5
"naive thymus-derived CD4-positive, alpha-beta T cell",17965,5


In [6]:
CELLS_PER_TYPE = 50   # small for learning; the plan uses up to 500
rng = np.random.default_rng(0)

chosen = np.sort(np.concatenate([
    rng.choice(group.soma_joinid.values, min(CELLS_PER_TYPE, len(group)), replace=False)
    for cell_type, group in cells[cells.cell_type.isin(kept.index)].groupby('cell_type')
]))
print(len(chosen), 'cells chosen')

2250 cells chosen


/var/folders/07/3l4k_4wn505bhm3d0hvndqqc0000gn/T/ipykernel_2462/3218943302.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for cell_type, group in cells[cells.cell_type.isin(kept.index)].groupby('cell_type')


## Step 5: Fetch the gene expression for only those cells

`get_anndata` returns a normal AnnData object, the same kind the app uses. `obs_coords` = which cells, `var_column_names` = which gene descriptions. `X` holds raw counts.

In [7]:
start = time.time()
with cellxgene_census.open_soma(census_version=CENSUS_VERSION) as census:
    adata = cellxgene_census.get_anndata(
        census, organism='Homo sapiens', obs_coords=chosen,
        obs_column_names=['cell_type', 'dataset_id', 'assay'],
        var_column_names=['feature_id', 'feature_name'],
    )
print(adata, f'\n\nfetched in {time.time() - start:.0f} s')

/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 2250 × 61497
    obs: 'cell_type', 'dataset_id', 'assay'
    var: 'feature_id', 'feature_name' 

fetched in 370 s


In [8]:
adata.obs.cell_type.value_counts()

cell_type
erythrocyte                                    50
classical monocyte                             50
CD8-positive, alpha-beta T cell                50
regulatory T cell                              50
gamma-delta T cell                             50
                                               ..
enterocyte                                      0
enterocyte of epithelium of large intestine     0
enterocyte of epithelium of small intestine     0
enterocyte of epithelium proper of duodenum     0
visceromotor neuron                             0
Name: count, Length: 898, dtype: int64

## Step 6: Save locally and record what you used

Saving means the app doesn't wait (or need the internet) the next time. The release and the query go with the file, so anyone can see exactly where it came from. (Saved to your home folder's Downloads here, outside the project, because `data/`-like files never go in the repo.)

In [9]:
from pathlib import Path

adata.uns['census'] = {'version': CENSUS_VERSION, 'query': query, 'cells_per_type': CELLS_PER_TYPE, 'seed': 0}
out = Path.home() / 'Downloads' / f'census_{TISSUE}_sample.h5ad'
adata.write_h5ad(out, compression='gzip')
print('saved', out)

saved /Users/aditivannela/Downloads/census_spleen_sample.h5ad


## What to remember about APIs

1. **Ask for as little as possible:** filter on the server, choose columns.
2. **Pin versions** of the package *and* the data.
3. **Expect failures:** the network can drop; real code shows a clear message (the app will offer "PanglaoDB only").
4. **Save results locally** so you pay the waiting cost once.

**Next (the app):** from this sample, compute each cell type's marker genes (Wilcoxon, as on the Marker Genes page) and test the clusters against them with the same over-representation test as PanglaoDB.

*Census data: CZ CELLxGENE Discover, CC BY 4.0; cite the Census when results are published.*